# 03 · 复购预测建模（v3）

**任务**：用客户在观察期内的行为，预测其在预测期内是否会再次购买。

**本 notebook 三件事**：
1. **行为特征工程**：订单级 AOV / 订单间隔 / 近期 30/60/90d 窗口 / log-transform → 21 个特征（v3 去掉了贡献 <0.005 AUC 的 SKU one-hot）
2. **XGBoost 调参 + early stopping**：验证集改为按 recency 时间切（不再随机）
3. **时间滚动验证**：3 个 rolling split × 3 个月 horizon 验证模型泛化

## 0. 加载

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import roc_curve, precision_recall_curve
from modeling import build_features, train_and_evaluate, top_k_capture_rate, time_rolling_validation

df = pd.read_parquet('../data/processed/transactions.parquet')
split_ts = pd.Timestamp('2011-06-01')
features, feat_cols = build_features(df, split_ts)
print(f'样本:     {len(features):,} 客户')
print(f'特征数:   {len(feat_cols)}   (行为聚合 + 订单间隔 + 近期 30/60/90d 窗口 + log)')
print(f'正样本率: {features["label"].mean():.1%}')
features.head()

样本:     4,996 客户
特征数:   21   (行为聚合 + 订单间隔 + 近期 30/60/90d 窗口 + log)
正样本率: 52.1%


,customer_id,invoice_count,total_amount,distinct_products,total_quantity,recency_days,tenure_days,avg_basket,return_order_count,return_rate,...,orders_last_30d,amount_last_30d,orders_last_60d,amount_last_60d,orders_last_90d,amount_last_90d,log_total_amount,log_avg_basket,log_amount_last_90d,label
0,12346,17,-64.68,30,52,133,400,6463.038333,5,0.294118,...,0.0,0.0,0.0,0.00,0.0,0.00,0.000000,8.774010,0.000000,0
1,12347,4,3146.75,90,1945,54,157,786.687500,0,0.000000,...,0.0,0.0,1.0,636.25,1.0,636.25,8.054443,6.669101,6.457162,1
2,12348,4,1709.40,25,2497,56,189,427.350000,0,0.000000,...,0.0,0.0,1.0,367.00,1.0,367.00,7.444483,6.059941,5.908083,1
3,12349,4,2646.99,92,988,215,327,890.380000,1,0.250000,...,0.0,0.0,0.0,0.00,0.0,0.00,7.881556,6.792771,0.000000,1
4,12350,1,334.40,17,197,118,0,334.400000,0,0.000000,...,0.0,0.0,0.0,0.00,0.0,0.00,5.815324,5.815324,0.000000,0


## 1. 特征分布 vs 标签（关键数值特征）

In [2]:
key = ['invoice_count', 'log_total_amount', 'avg_gap_days', 'recency_days',
       'orders_last_90d', 'distinct_products']
melted = features.melt(id_vars='label', value_vars=key)
fig = px.box(melted, x='variable', y='value', color='label',
             title='特征 × 标签（0=未复购 / 1=复购）· log-y 轴', log_y=True)
fig.update_layout(height=440, xaxis_title='', yaxis_title='取值 (log)')
fig.show()

> **眼见为实**：复购客户（label=1）的 `orders_last_90d`、`invoice_count`、`log_total_amount` 显著更高，`recency_days` 和 `avg_gap_days` 更低。新加的近期窗口 + 订单间隔与直觉一致。

## 2. 三模型对比（单一主切 · 6 个月 horizon）

In [3]:
results = train_and_evaluate(features, feat_cols)
summary = pd.DataFrame([
    {'模型': r.name,
     'ROC-AUC': round(r.auc, 3),
     'Top 10% 覆盖': f'{top_k_capture_rate(r.y_test, r.y_proba, 0.1):.1%}',
     'Top 20% 覆盖': f'{top_k_capture_rate(r.y_test, r.y_proba, 0.2):.1%}'}
    for r in results])
summary

,模型,ROC-AUC,Top 10% 覆盖,Top 20% 覆盖
0,LogisticRegression,0.819,18.4%,34.7%
1,RandomForest,0.806,18.4%,35.2%
2,XGBoost (tuned),0.812,18.1%,35.3%


> 这个 horizon 下正样本率高达 52%，本身偏容易——AUC 提升空间有限，三模型表现基本持平。
> 真正值得看的故事在下一节：**时间滚动验证**，horizon 缩短到 3 个月（正样本率降到 32-35%，更贴近生产）。

## 3. 时间滚动验证 — 3 个 rolling folds × 3 个月 horizon

In [4]:
rolling = time_rolling_validation(
    df,
    split_points=[pd.Timestamp('2011-01-01'), pd.Timestamp('2011-04-01'), pd.Timestamp('2011-07-01')],
    label_horizon_days=90,
)
rolling

,fold_split,label_end,n_samples,positive_rate,model,auc,top10_capture,top20_capture
0,2011-01-01,2011-04-01,4411,0.318,LogisticRegression,0.789,0.248,0.430
1,2011-01-01,2011-04-01,4411,0.318,RandomForest,0.774,0.268,0.442
2,2011-01-01,2011-04-01,4411,0.318,XGBoost (tuned),0.777,0.222,0.447
3,2011-04-01,2011-06-30,4783,0.347,LogisticRegression,0.819,0.243,0.472
4,2011-04-01,2011-06-30,4783,0.347,RandomForest,0.804,0.253,0.448
5,2011-04-01,2011-06-30,4783,0.347,XGBoost (tuned),0.815,0.243,0.460
6,2011-07-01,2011-09-29,5104,0.336,LogisticRegression,0.824,0.249,0.469
7,2011-07-01,2011-09-29,5104,0.336,RandomForest,0.817,0.268,0.448
8,2011-07-01,2011-09-29,5104,0.336,XGBoost (tuned),0.823,0.261,0.462


In [5]:
# 三模型在三个 fold 上的 AUC 对比
fig = px.bar(rolling, x='fold_split', y='auc', color='model', barmode='group',
             text=rolling['auc'].apply(lambda v: f'{v:.3f}'),
             title='各 fold × 各模型 AUC',
             labels={'fold_split': 'Fold 观察期截止日', 'auc': 'ROC-AUC'})
fig.update_layout(height=400, yaxis_range=[0.7, 0.85])
fig.update_traces(textposition='outside')
fig.show()

mean_auc = rolling.groupby('model')['auc'].mean().round(3)
print('平均 AUC:'); print(mean_auc)

平均 AUC:
model
LogisticRegression    0.811
RandomForest          0.798
XGBoost (tuned)       0.805
Name: auc, dtype: float64


> **结论**：三模型在滚动验证下 AUC 都在 **0.77-0.82** 区间，**Logistic Regression 与 XGBoost 基本持平，Random Forest 略低**。这说明：（1）信号大部分是线性可捕获的，（2）模型在不同时间窗口上表现一致，**泛化性 OK，不是过拟合某一时间段**。生产里选 LR 的可解释性 + XGB 的非线性兜底都合理。

## 4. ROC & Precision-Recall 双视图（主切最佳模型）

In [6]:
best = max(results, key=lambda r: r.auc)
fpr, tpr, _ = roc_curve(best.y_test, best.y_proba)
prec, rec, _ = precision_recall_curve(best.y_test, best.y_proba)
fig = make_subplots(rows=1, cols=2, subplot_titles=(f'ROC (AUC={best.auc:.3f})', 'Precision-Recall'))
fig.add_trace(go.Scatter(x=fpr, y=tpr, name=best.name, line=dict(width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash='dash', color='gray'), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=rec, y=prec, name=best.name, showlegend=False, line=dict(width=3)), row=1, col=2)
fig.update_layout(height=400)
fig.show()

## 5. Gain & Lift 曲线 — 业务价值

In [7]:
def gain_lift(y_true, y_proba):
    order = np.argsort(y_proba)[::-1]
    y_sorted = np.array(y_true)[order]
    cum = np.cumsum(y_sorted) / y_sorted.sum()
    pct = np.arange(1, len(y_sorted) + 1) / len(y_sorted)
    return pct, cum

pct, gain = gain_lift(best.y_test, best.y_proba)
lift = gain / pct
fig = make_subplots(rows=1, cols=2, subplot_titles=('累计 Gain', 'Lift'))
fig.add_trace(go.Scatter(x=pct, y=gain, name='模型', line=dict(width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash='dash', color='gray'), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=pct, y=lift, showlegend=False, line=dict(width=3)), row=1, col=2)
fig.add_hline(y=1, line_dash='dash', line_color='gray', row=1, col=2)
fig.update_xaxes(tickformat='.0%')
fig.update_yaxes(tickformat='.0%', row=1, col=1)
fig.update_layout(height=400)
fig.show()

> **业务读法**：按模型打分从高到低排序，**3 个月 horizon 下 Top 10% 覆盖 25-27% 的真实复购者（lift ≈ 2.5x）**，Top 20% 覆盖 44-47%（lift ≈ 2.3x）。相同预算下触达高价值客户的效率显著高于随机投放。

## 6. 特征重要度 · Top 20

In [8]:
rf = next(r for r in results if r.name == 'RandomForest').model
imp = (pd.DataFrame({'feature': feat_cols, 'importance': rf.feature_importances_})
       .sort_values('importance', ascending=False).head(20)
       .sort_values('importance'))
fig = px.bar(imp, x='importance', y='feature', orientation='h',
             color='importance', color_continuous_scale='Blues',
             title='Random Forest · Top 20 特征重要度')
fig.update_layout(height=520, coloraxis_showscale=False)
fig.show()

> **洞察**：`recency_days` / `invoice_count` / `orders_last_90d` / `avg_gap_days` 占据前列——**行为频次与节奏**比消费金额更能预测下次购买。这也解释了为什么 v2 里加的 Top-15 SKU one-hot 在 ablation 测试中只带来 <0.005 AUC 提升（已在 v3 移除）：礼品批发业务里「什么时候买、买多勤」比「买了什么」更能预测下次行为。

## 7. 输出 Top 10% 高潜名单 → 对接 CRM

In [9]:
best_full = next(r for r in results if r.name == best.name).model
all_features = features.copy()
# 用 XGBoost 或 RF 直接 predict_proba，和训练时一致（未标准化）
if best.name == 'LogisticRegression':
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler().fit(features[feat_cols].values)
    scores = best_full.predict_proba(scaler.transform(features[feat_cols].values))[:, 1]
else:
    scores = best_full.predict_proba(features[feat_cols].values)[:, 1]
all_features['score'] = scores
top10 = all_features.sort_values('score', ascending=False).head(int(len(all_features) * 0.1))
print(f'Top 10% 高潜名单: {len(top10)} 位客户')
print(f'名单平均历史消费:  £{top10["total_amount"].mean():,.0f}')
print(f'名单平均 Recency:  {top10["recency_days"].mean():.0f} 天')
print(f'名单复购率真值:    {top10["label"].mean():.1%}  （基线: {all_features["label"].mean():.1%}）')
top10[['customer_id', 'invoice_count', 'total_amount', 'recency_days', 'orders_last_90d', 'score']].head(10)

Top 10% 高潜名单: 499 位客户
名单平均历史消费:  £12,231
名单平均 Recency:  16 天
名单复购率真值:    97.0%  （基线: 52.1%）


,customer_id,invoice_count,total_amount,recency_days,orders_last_90d,score
308,12748,231,30220.85,0,44.0,1.000000
2133,14911,336,171749.87,5,46.0,1.000000
4623,17841,185,40530.38,0,35.0,1.000000
594,13089,193,82519.20,5,35.0,1.000000
1867,14606,198,25031.27,2,28.0,0.999985
1096,13694,134,154501.05,0,19.0,0.999964
1190,13798,99,58822.24,4,23.0,0.999955
2479,15311,210,83133.35,4,24.0,0.999883
490,12971,72,9151.20,4,21.0,0.999864
3411,16422,111,44354.23,4,24.0,0.999776


---

## 小结（v3）

- **特征工程** 9 维 → 21 维，订单间隔 + 近期窗口是高 ROI 组合；v3 修掉了 avg_basket / return_count / avg_gap_days 三处语义 bug，并移除了贡献边际的 SKU one-hot
- **三模型在滚动验证中 AUC 0.77-0.82 基本持平**，Logistic Regression 的表现与 XGBoost 相当——信号以线性为主
- **时间滚动验证** 3 个 fold 表现一致，泛化性得到验证（而非单次随机切的运气）
- **业务价值**：3 个月 horizon 下 Top 10% 覆盖 25-27% 复购者（lift ≈ 2.5-2.7x），Top 10% 名单可直接落到召回动作
- **局限与下一步**：加入用户画像 / 品类聚合特征、做成本敏感的阈值调优、对接真实 A/B